<a href="https://colab.research.google.com/github/angelinetipa/deped-data-audit/blob/main/deped_audit.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [2]:
# ================================================================
# MOUNT GOOGLE DRIVE — allows Colab to read/write files in Drive
# so data and outputs persist after the session ends.
# Running this will prompt a Google sign-in/permission request —
# click "Connect to Google Drive" and grant access.
# ================================================================
from google.colab import drive
drive.mount('/content/drive', force_remount=True)

Mounted at /content/drive


In [3]:
# Load the raw data fresh, then keep an untouched copy in memory.
# df_original is never modified — if a fix goes wrong, reset with
# df = df_original.copy() instead of reloading from Drive.
RAW = "/content/drive/MyDrive/deped-audit/data/raw.xlsx"

import pandas as pd
df = pd.read_excel(RAW, sheet_name="DB", header=4)
df_original = df.copy()

print(df.shape)  # should print (60167, 72)

(60167, 72)


In [ ]:
# Auto-discovery, Part A (v4): split into two buckets, no stopword
# filtering. Bucket 2 will include some real short words alongside
# actual abbreviations; these are reviewed manually rather than
# filtered automatically.

import re
from collections import Counter

# Bucket 1: tokens containing "." or "/". Punctuation is treated as
# a structural signal of an abbreviation (e.g. "St.", "E/S"), and
# produces very few false positives.
def find_punctuated(series, max_letters=5, min_count=5):
    tokens = Counter()
    for text in series.dropna().astype(str):
        for word in text.split():
            has_mark = "." in word or "/" in word
            letters_only = re.sub(r"[^A-Za-z]", "", word)
            if has_mark and 1 <= len(letters_only) <= max_letters:
                tokens[word] += 1
    # Only patterns meeting the minimum count are kept, so one-off
    # typos are separated from recurring patterns.
    return Counter({w: c for w, c in tokens.items() if c >= min_count})

# Bucket 2: short ALL-CAPS tokens with no punctuation (e.g. "ES",
# "NHS"). No stopword list is applied, so some real short words
# (e.g. roman numerals, "SAN") will also appear here and require
# a manual pass to separate from genuine abbreviations.
def find_caps_no_punct(series, max_letters=5, min_count=5):
    tokens = Counter()
    for text in series.dropna().astype(str):
        for word in text.split():
            clean = re.sub(r"[^A-Za-z]", "", word)
            if (
                word.isupper()
                and "." not in word and "/" not in word
                and 1 <= len(clean) <= max_letters
            ):
                tokens[word] += 1
    return Counter({w: c for w, c in tokens.items() if c >= min_count})

# Bucket 1 result for School Name. Expected to surface items such
# as "St.", "Inc.", "E/S", "P/S".
print("School Name — punctuation-marked abbreviations")
for word, count in find_punctuated(df["School Name"]).most_common(40):
    print(f"{word}: {count}")

# Bucket 2 result for School Name. Expected to surface items such
# as "ES", "NHS", "CS", "CES", alongside some real short words that
# require manual filtering.
print("\nSchool Name — ALL-CAPS short tokens (manual review)")
for word, count in find_caps_no_punct(df["School Name"]).most_common(30):
    print(f"{word}: {count}")

# Bucket 1 result for Street Address. Expected to surface items
# such as "Brgy.", "St.", "Sta.", "Subd.".
print("\nStreet Address — punctuation-marked abbreviations")
for word, count in find_punctuated(df["Street Address"]).most_common(30):
    print(f"{word}: {count}")

# Bucket 2 result for Street Address. Expected to surface items
# such as "PUROK", "SITIO", "SAN", alongside some real short words
# that require manual filtering.
print("\nStreet Address — ALL-CAPS short tokens (manual review)")
for word, count in find_caps_no_punct(df["Street Address"]).most_common(30):
    print(f"{word}: {count}")

School Name — punctuation-marked abbreviations
Inc.: 5826
St.: 963
Sta.: 783
Sr.: 443
Sto.: 421
INC.: 387
A.: 229
M.: 194
P.: 193
C.: 175
B.: 167
L.: 155
Elem.: 142
R.: 138
E.: 125
F.: 125
G.: 120
Dr.: 118
S.: 116
T.: 115
Mem.: 104
D.: 102
V.: 99
E/S: 87
Gen.: 76
Sch.: 74
Ext.: 70
J.: 66
Mt.: 50
H.: 48
Jr.: 42
ST.: 34
N.: 34
O.: 33
Gov.: 32
Bo.: 31
Corp.: 28
I.: 24
Pres.: 22
Inc.,: 20

School Name — ALL-CAPS short tokens (manual review)
ES: 16692
NHS: 2088
PS: 1397
CS: 440
CES: 435
HS: 343
HIGH: 252
II: 238
I: 233
OF: 177
MES: 176
IS: 175
ES): 164
MS: 131
SPED: 123
SDA: 82
NHS): 67
MNHS: 63
STI: 63
UCCP: 60
AMA: 52
INC: 51
SAN: 48
SHS: 43
III: 40
IP: 40
AND: 39
MHS: 29
A: 28
PS): 27

Street Address — punctuation-marked abbreviations
Brgy.: 4161
St.: 2762
St.,: 1579
Sta.: 805
Sto.: 407
Subd.: 275
P.: 256
ST.: 238
BRGY.: 227
Prk.: 227
A.: 208
Ave.,: 204
Subd.,: 189
Ave.: 175
Gen.: 173
M.: 166
-Brgy.: 160
J.: 157
Blk.: 150
F.: 148
-n/a: 145
st.: 141
Sor.: 136
Bldg.,: 133
So.: 132
E.: 131


In [ ]:
# Auto-discovery, Part B: detect leading non-letter characters.
# The first character of each value is checked. If it is not a
# letter or digit, it is treated as likely formatting noise
# (dashes, hashes, quotes) rather than real data content.

from collections import Counter

def find_leading_chars(series):
    chars = Counter()
    for text in series.dropna().astype(str):
        if text and not text[0].isalnum():
            chars[text[0]] += 1
    return chars

# Counts for Street Address, printed one line per character so
# each result is readable without scanning a single dense row.
print("Street Address — leading non-letter characters")
for char, count in find_leading_chars(df["Street Address"]).most_common():
    print(f"{char!r}: {count}")

# Same check for School Name.
print("\nSchool Name — leading non-letter characters")
for char, count in find_leading_chars(df["School Name"]).most_common():
    print(f"{char!r}: {count}")

Street Address — leading non-letter characters
'-': 5724
'#': 469
'`': 22
'(': 18
'*': 8
'.': 6
',': 5
'_': 4
'=': 3
'"': 2
"'": 1
'@': 1
'\xa0': 1

School Name — leading non-letter characters
'(': 5
'"': 1


In [ ]:
# Auto-discovery, Part C: detect placeholder-style blanks.
# Real free-text values (like street addresses) are highly varied
# across 60,000+ rows, so a value repeating far more often than
# expected is likely a placeholder standing in for missing data,
# not a genuine repeated address.

print("Street Address — most repeated values")
for value, count in df["Street Address"].value_counts().head(30).items():
    print(f"{value!r}: {count}")

print("\nBarangay — most repeated values")
for value, count in df["School Name"].value_counts().head(30).items():
    print(f"{value!r}: {count}")

Street Address — most repeated values
'-': 2375
'Purok 1': 465
'Purok 2': 432
'none': 413
'Purok 3': 357
'National Highway': 299
'Poblacion': 277
'Purok 4': 202
'not applicable': 181
'Purok 5': 172
'National Road': 169
'-n/a': 145
'Provincial Road': 128
'-none': 125
'0': 108
'Brgy Road': 106
'Not Applicable': 100
'Centro': 98
'Maharlika Highway': 97
'Purok 6': 84
'Zone 1': 80
'-None': 77
'Zone 3': 73
'Proper': 69
'Purok 7': 63
'San Isidro': 62
'NONE': 61
'Zone 2': 61
'PUROK 1': 58
'Brgy. Road': 54

Barangay — most repeated values
'San Isidro Elementary School': 108
'San Isidro ES': 106
'San Jose ES': 89
'San Jose Elementary School': 76
'San Vicente ES': 76
'San Roque Elementary School': 75
'San Antonio ES': 67
'San Roque ES': 64
'San Vicente Elementary School': 63
'Sta. Cruz ES': 55
'San Juan ES': 55
'Sta. Cruz Elementary School': 54
'San Juan Elementary School': 53
'San Antonio Elementary School': 49
'San Miguel Elementary School': 48
'Magsaysay ES': 48
'Salvacion Elementary School': 

In [ ]:
# Auto-discovery, Part D (continued): check casing on free-text
# fields specifically. Barangay and Province returned no variants,
# consistent with being drawn from a fixed reference list. School
# Name and Street Address are typed manually per school, so casing
# inconsistency is expected to surface here instead.

def find_casing_variants(series, min_count=2):
    variants = {}
    for text in series.dropna().astype(str):
        key = text.lower()
        variants.setdefault(key, set()).add(text)
    return {k: v for k, v in variants.items() if len(v) > 1}

print("School Name — casing variants")
results = find_casing_variants(df["School Name"])
for key, forms in sorted(results.items(), key=lambda x: -len(x[1]))[:20]:
    print(f"{key}: {forms}")

print("\nStreet Address — casing variants")
results = find_casing_variants(df["Street Address"])
for key, forms in sorted(results.items(), key=lambda x: -len(x[1]))[:20]:
    print(f"{key}: {forms}")

School Name — casing variants
san isidro elementary school: {'San Isidro Elementary School', 'SAN ISIDRO ELEMENTARY SCHOOL', 'San Isidro Elementary SChool', 'San Isidro Elementary school'}
bliss elementary school: {'BLISS Elementary School', 'BLISS ELEMENTARY SCHOOL', 'Bliss Elementary School'}
pag-asa elementary school: {'Pag-Asa Elementary School', 'Pag-asa Elementary School', 'PAG-ASA ELEMENTARY SCHOOL'}
sto. niño es: {'STO. NIÑO ES', 'Sto. Niño ES', 'Sto. NiÑo ES'}
ilian elementary school: {'Ilian Elementary school', 'Ilian Elementary School', 'ILIAN ELEMENTARY SCHOOL'}
cogon elementary school: {'Cogon Elementary School', 'COGON ELEMENTARY SCHOOL', 'Cogon Elementary school'}
kalilangan elementary school: {'Kalilangan Elementary School', 'KALILANGAN ELEMENTARY SCHOOL', 'Kalilangan Elementary school'}
special education center: {'SPECIAL EDUCATION CENTER', 'Special Education Center'}
banban elementary school: {'BANBAN ELEMENTARY SCHOOL', 'Banban Elementary School'}
suyo national high 

In [ ]:
# Auto-discovery, Part E: detect encoding artifacts (mojibake).
# Text originally saved as UTF-8 but read back as Latin-1 produces
# garbled sequences such as "Ã" or "Â" in place of accented
# characters (e.g. Ñ). Presence of these sequences is checked
# directly, rather than assuming which fields are affected.

def find_mojibake(series):
    count = 0
    examples = []
    for text in series.dropna().astype(str):
        if "Ã" in text or "Â" in text:
            count += 1
            if len(examples) < 10:
                examples.append(text)
    return count, examples

for col in ["School Name", "Street Address", "Province", "Municipality", "Barangay"]:
    count, examples = find_mojibake(df[col])
    print(f"{col}: {count} rows affected")
    for ex in examples:
        print(f"  {ex!r}")
    print()

School Name: 12 rows affected
  'Sto. Nino Ilaya NHS (Formerly San Francisco B NHS Sto. NiÃ±o Ilaya Ext.)'
  'BaÃ±adero High School'
  'Marcial O. RaÃ±ola Memorial School'
  'Tinorongan National High School (Formerly SagÃ±ay Western  HS)'
  'Concepcion PequeÃ±a National High School'
  'Florentina F. CaÃ±a Recto NHS'
  'DoÃ±a E.J. Garcia ES'
  'Bartolome & Manuela PaÃ±ares MNHS'
  'DoÃ±a Milagros MNHS'
  'DoÃ±a Liling Neis Negapatan NHS'

Street Address: 6 rows affected
  'PASADEÃ‘A'
  'Nato, SagÃ±ay'
  '30 de Deciembre,F. MascariÃ±as Zone 12'
  'DOÃ‘A JOSEFA'
  '3 Lopez Jaena St., TaÃ±ong'
  'Agricultores St., Sto. NiÃ±o'

Province: 0 rows affected

Municipality: 0 rows affected

Barangay: 260 rows affected
  'SABAÃ‘GAN (MARCOS)'
  'SAPA PEQUEÃ‘A'
  'OSMEÃ‘A (POB.)'
  'MALACAÃ‘ANG'
  'MALACAÃ‘ANG'
  'PIÃ‘A ESTE'
  'PIÃ‘A WESTE'
  'BITAG PEQUEÃ‘O'
  'LOURDES (EL ESCAÃ‘O)'
  'CUTOG PEQUEÃ‘O'



In [ ]:
# Auto-discovery, Part F: detect extra internal whitespace.
# Values containing a double space, or leading/trailing spaces,
# are checked separately from leading junk characters (Part B),
# since whitespace does not match the non-letter check used there.
# Two conditions are counted: double spaces anywhere in the text,
# and spaces at the very start or end of the value.

import re

def find_double_spaces(series):
    return series.dropna().astype(str).str.contains(r"  ").sum()

def find_edge_spaces(series):
    return series.dropna().astype(str).apply(lambda t: t != t.strip()).sum()

for col in ["School Name", "Street Address", "Province", "Municipality", "Barangay"]:
    double = find_double_spaces(df[col])
    edge = find_edge_spaces(df[col])
    print(f"{col}: {double} rows with double spaces, {edge} rows with leading/trailing spaces")

School Name: 460 rows with double spaces, 1 rows with leading/trailing spaces
Street Address: 556 rows with double spaces, 7 rows with leading/trailing spaces
Province: 2720 rows with double spaces, 324 rows with leading/trailing spaces
Municipality: 221 rows with double spaces, 0 rows with leading/trailing spaces
Barangay: 457 rows with double spaces, 0 rows with leading/trailing spaces


## **Technical corrections**

In [4]:
# Fix, Step 1: repair mojibake (Ñ/ñ decoded incorrectly).
# Applied in place, since this repairs a technical encoding error
# rather than reinterpreting legitimate data.

def fix_mojibake(text):
    if not isinstance(text, str):
        return text
    if "Ã" in text or "Â" in text:
        try:
            return text.encode("latin-1").decode("utf-8")
        except (UnicodeEncodeError, UnicodeDecodeError):
            return text
    return text

TEXT_COLS = ["School Name", "Street Address", "Province", "Municipality", "Barangay"]

# Before-and-after examples are printed for a sample of affected
# rows, so the fix can be checked by eye before trusting it at scale.
for col in TEXT_COLS:
    before = df[col].copy()
    df[col] = df[col].map(fix_mojibake)
    changed = before != df[col]
    if changed.sum() > 0:
        print(f"{col}: {changed.sum()} rows fixed")
        for b, a in list(zip(before[changed], df[col][changed]))[:5]:
            print(f"  {b!r}  ->  {a!r}")
        print()

School Name: 12 rows fixed
  'Sto. Nino Ilaya NHS (Formerly San Francisco B NHS Sto. NiÃ±o Ilaya Ext.)'  ->  'Sto. Nino Ilaya NHS (Formerly San Francisco B NHS Sto. Niño Ilaya Ext.)'
  'BaÃ±adero High School'  ->  'Bañadero High School'
  'Marcial O. RaÃ±ola Memorial School'  ->  'Marcial O. Rañola Memorial School'
  'Tinorongan National High School (Formerly SagÃ±ay Western  HS)'  ->  'Tinorongan National High School (Formerly Sagñay Western  HS)'
  'Concepcion PequeÃ±a National High School'  ->  'Concepcion Pequeña National High School'

Street Address: 1686 rows fixed
  nan  ->  nan
  nan  ->  nan
  nan  ->  nan
  nan  ->  nan
  nan  ->  nan

Barangay: 70 rows fixed
  nan  ->  nan
  nan  ->  nan
  nan  ->  nan
  nan  ->  nan
  nan  ->  nan

